## **Finland labor market gig shift 2019-2026**

In [1]:
import numpy as np
import pandas as pd

## 1. Traficom Taxi Licences (2016–2023)
Source: Finnish Transport and Communications Agency (Traficom)
Issues to fix:
- Semicolon separator instead of comma
- Thousands separator spaces in licence counts e.g. "9 624" → 9624
- Date column needs splitting into year and month

In [2]:
# Load raw file 
df_taxi = pd.read_csv('../data/raw/traficom_taxi_licences_2016_2023.csv', sep=';')

In [3]:
df_taxi.shape

(9, 2)

In [4]:
df_taxi.head()

,Date,Number of licenses
0,12/2016,9 624
1,12/2017,9 499
2,6/2018,9 487
3,12/2018,11 852
4,12/2019,12 332


In [5]:
df_taxi.info()

<class 'pandas.DataFrame'>
RangeIndex: 9 entries, 0 to 8
Data columns (total 2 columns):
 #   Column              Non-Null Count  Dtype
---  ------              --------------  -----
 0   Date                9 non-null      str  
 1   Number of licenses  9 non-null      str  
dtypes: str(2)
memory usage: 276.0 bytes


In [6]:
df_taxi.isnull().sum()

Date                  0
Number of licenses    0
dtype: int64

In [7]:
# Fix column names to lowercase with underscores
df_taxi.columns = ['date', 'number_of_licences']

In [8]:
# Convert date to datetime
df_taxi['date'] = pd.to_datetime(df_taxi['date'], format='%m/%Y')

In [9]:
df_taxi['date'].dtype

dtype('<M8[us]')

In [10]:
# Remove thousands separator spaces and convert to integer
df_taxi['number_of_licences'] = df_taxi['number_of_licences'].str.replace(' ', '').astype(int)

In [11]:
# Extract year and month as separate columns
df_taxi['year'] =  df_taxi['date'].dt.year
df_taxi['month']= df_taxi['date'].dt.month

In [12]:
# Add source column for traceability in PostgreSQL later
df_taxi['source'] = 'traficom'

In [13]:
df_taxi.head()

,date,number_of_licences,year,month,source
0,2016-12-01,9624,2016,12,traficom
1,2017-12-01,9499,2017,12,traficom
2,2018-06-01,9487,2018,6,traficom
3,2018-12-01,11852,2018,12,traficom
4,2019-12-01,12332,2019,12,traficom


In [14]:
df_taxi.dtypes

date                  datetime64[us]
number_of_licences             int64
year                           int32
month                          int32
source                           str
dtype: object

In [15]:
# Export cleaned dataframe to CSV for PostgreSQL import

df_taxi.to_csv('../data/cleaned/traficom_taxi_licences_2016_2023.csv', index=False)
print("Exported Succesfully")

Exported Succesfully


## 2. StatFin Population by Activity Type (1987–2024)
Source: Statistics Finland (stat.fi)
Issues to fix:
- Wide format (years as columns) needs reshaping to long format (one row per year)
- First row is a header artifact from stat.fi export
- Values are percentages of total population

In [16]:
# Load raw file without headers since first rows contain metadata, not column names

df_population = pd.read_csv('../data/raw/statfin_population_by_activity_1987_2024.csv', header=None)

In [17]:
df_population

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,Population by main type of activity by municip...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Whole country,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1987.0,1988.0,1989.0,1990.0,1991.0,1992.0,1993.0,1994.0,1995.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0
3,Employed,47.0,47.5,47.7,46.7,43.1,39.9,37.0,37.6,37.8,...,41.1,41.4,42.2,43.0,43.0,41.3,42.8,43.6,43.1,43.0
4,Unemployed,3.0,2.6,2.2,2.8,6.0,8.8,10.5,9.6,9.3,...,6.8,6.5,5.4,4.6,4.7,6.4,4.9,4.6,5.3,5.7
5,0-14 years old,19.3,19.4,19.3,19.3,19.2,19.2,19.1,19.1,19.0,...,16.3,16.2,16.2,16.0,15.8,15.6,15.4,15.1,14.9,14.6
6,"Students, pupils",6.3,6.2,6.4,6.6,7.1,7.4,8.4,8.7,8.9,...,7.5,7.4,7.3,7.1,7.3,7.6,7.6,7.4,7.6,7.9
7,"Conscripts, persons in non-military service",0.6,0.5,0.4,0.6,0.4,0.3,0.4,0.4,0.3,...,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1,0.1
8,Pensioners,20.3,20.4,20.6,20.8,20.9,21.0,21.3,21.5,21.6,...,24.9,25.3,25.7,25.9,26.0,26.0,26.0,26.0,25.8,25.6
9,Other persons outside the labour force,3.6,3.4,3.3,3.3,3.2,3.4,3.3,3.2,3.2,...,3.1,3.1,3.2,3.2,3.2,3.1,3.1,3.2,3.3,3.1


In [18]:
df_population.shape

(12, 39)

In [19]:
df_population.info()

<class 'pandas.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 39 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   0       11 non-null     str    
 1   1       8 non-null      float64
 2   2       8 non-null      float64
 3   3       8 non-null      float64
 4   4       8 non-null      float64
 5   5       8 non-null      float64
 6   6       8 non-null      float64
 7   7       8 non-null      float64
 8   8       8 non-null      float64
 9   9       8 non-null      float64
 10  10      8 non-null      float64
 11  11      8 non-null      float64
 12  12      8 non-null      float64
 13  13      8 non-null      float64
 14  14      8 non-null      float64
 15  15      8 non-null      float64
 16  16      8 non-null      float64
 17  17      8 non-null      float64
 18  18      8 non-null      float64
 19  19      8 non-null      float64
 20  20      8 non-null      float64
 21  21      8 non-null      float64
 22  22      8 non-n

In [20]:
df_population.head()

,0,1,2,3,4,5,6,7,8,9,...,29,30,31,32,33,34,35,36,37,38
0,Population by main type of activity by municip...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Whole country,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,1987.0,1988.0,1989.0,1990.0,1991.0,1992.0,1993.0,1994.0,1995.0,...,2015.0,2016.0,2017.0,2018.0,2019.0,2020.0,2021.0,2022.0,2023.0,2024.0
3,Employed,47.0,47.5,47.7,46.7,43.1,39.9,37.0,37.6,37.8,...,41.1,41.4,42.2,43.0,43.0,41.3,42.8,43.6,43.1,43.0
4,Unemployed,3.0,2.6,2.2,2.8,6.0,8.8,10.5,9.6,9.3,...,6.8,6.5,5.4,4.6,4.7,6.4,4.9,4.6,5.3,5.7


In [21]:
# Extract years from row 2 (skip first cell which is NaN)
years = df_population.iloc[2, 1:].tolist()
years = [int(float(y)) for y in years]

In [22]:
# Extract categories and values from rows 3 to 9
categories = df_population.iloc[3:10, 0].tolist()
values = df_population.iloc[3:10, 1:].values

In [23]:
# Build clean long format dataframe
rows = []
for i, category in enumerate(categories):
    for j, year in enumerate(years):
        rows.append({
            'year': year,
            'category': category,
            'value_pct': values[i][j],
            'source': 'stat.fi'
        })

df_population = pd.DataFrame(rows)

In [24]:
    # Filter to 2019-2024
df_population = df_population[df_population['year'] >= 2019]

In [25]:
df_population.head()

,year,category,value_pct,source
32,2019,Employed,43.0,stat.fi
33,2020,Employed,41.3,stat.fi
34,2021,Employed,42.8,stat.fi
35,2022,Employed,43.6,stat.fi
36,2023,Employed,43.1,stat.fi


In [26]:
# Export cleaned dataframe to CSV for PostgreSQL import

df_population.to_csv('../data/cleaned/statfin_population_by_activity_1987_2024.csv', index=False)

## 3. StatFin Employment Rate Monthly (2010–2026)
Source: Statistics Finland (stat.fi)
Issues to fix:
- No proper column names — first row contains month labels, not headers
- Date format is '2010M01' — needs splitting into separate year and month columns
- Rows 3 and 4 are junk (Unit and Source footnotes) — need to be dropped
- Wide format — months spread across 196 columns, needs reshaping to one row per month

In [179]:
# Load raw file without headers since first rows contain metadata, not column names

df_employment = pd.read_csv('../data/raw/statfin_employment_rate_monthly_2010_2026.csv',header=None)

In [180]:

df_employment.head()

,0,1,2,3,4,5,6,7,8,9,...,187,188,189,190,191,192,193,194,195,196
0,Employment rate and trend of employment rate o...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,2010M01,2010M02,2010M03,2010M04,2010M05,2010M06,2010M07,2010M08,2010M09,...,2025M07,2025M08,2025M09,2025M10,2025M11,2025M12,2026M01,2026M02,2026M03,2026M04
2,Employment rate,69.5,70.7,70.8,70.8,72.9,73.1,73.1,73.4,71.4,...,76.5,76.2,75.9,76.3,75.6,75.0,74.8,75.3,74.2,74.7
3,"Employment rate, trend",71.6,71.6,71.6,71.6,71.7,71.7,71.7,71.8,71.8,...,75.8,75.9,75.9,75.9,75.9,75.7,75.7,75.6,75.4,75.3
4,Unit: %,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [181]:
## Extract row 0 to column header
df_employment.columns = ['category'] + df_employment.iloc[1, 1:].tolist()

## Drop  row 0 since it's now a header
df_employment = df_employment.drop(index=0)

In [182]:
df_employment.head()

,category,2010M01,2010M02,2010M03,2010M04,2010M05,2010M06,2010M07,2010M08,2010M09,...,2025M07,2025M08,2025M09,2025M10,2025M11,2025M12,2026M01,2026M02,2026M03,2026M04
1,NaN,2010M01,2010M02,2010M03,2010M04,2010M05,2010M06,2010M07,2010M08,2010M09,...,2025M07,2025M08,2025M09,2025M10,2025M11,2025M12,2026M01,2026M02,2026M03,2026M04
2,Employment rate,69.5,70.7,70.8,70.8,72.9,73.1,73.1,73.4,71.4,...,76.5,76.2,75.9,76.3,75.6,75.0,74.8,75.3,74.2,74.7
3,"Employment rate, trend",71.6,71.6,71.6,71.6,71.7,71.7,71.7,71.8,71.8,...,75.8,75.9,75.9,75.9,75.9,75.7,75.7,75.6,75.4,75.3
4,Unit: %,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,"Source: Statistics Finland, labour force survey",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [183]:
# Drop junk rows 3 and 4
df_employment =df_employment.drop(index=[1,4,5])

In [184]:
# Reshape from wide to long format
# Each month column becomes a row, creating 'month_code' and 'rate' columns

df_employment = df_employment.melt(id_vars='category',
                                   var_name='month_code',
                                   value_name='rate')

In [186]:
# Split '2010M01' into separate year and month integer columns
df_employment['year'] = df_employment['month_code'].str[:4].astype(int)
df_employment['month'] = df_employment['month_code'].str[5:].astype(int)



In [187]:
# Drop month_code as it's no longer needed — year and month columns replace it
df_employment = df_employment.drop(columns='month_code')

In [188]:
# Convert rate to float
df_employment['rate'] = df_employment['rate'].astype(float)

In [189]:
# Filter to 2019 onwards
df_employment =df_employment[df_employment['year'] >= 2019]

In [190]:
## # Add source column
df_employment['source'] = 'stat.fi'

In [ ]:
# Export cleaned dataframe to CSV for PostgreSQL import

df_employment.to_csv('../data/cleaned/statfin_employment_rate_monthly_2010_2026.csv', index=False)

## 4. Eurostat Unemployment Monthly (EU, 2019–2025)
Source: Eurostat
Issues to fix:
- Many unnecessary columns — keeping only country code, country name, date, and unemployment rate
- Column names are long and messy — renaming to clean lowercase
- Contains all available dates — filtering to 2019 onwards
- TIME_PERIOD format is '2025-07' — splitting into separate year and month columns

In [93]:
df_eurostat=pd.read_csv('../data/raw/eurostat_unemployment_monthly_eu_2025.csv')

In [92]:
df_eurostat.head()

,STRUCTURE,STRUCTURE_ID,STRUCTURE_NAME,freq,Time frequency,s_adj,Seasonal adjustment,age,Age class,unit,Unit of measure,sex,Sex,geo,Geopolitical entity (reporting),TIME_PERIOD,OBS_VALUE
0,dataflow,ESTAT:UNE_RT_M$DEFAULTVIEW(1.0),Unemployment by sex and age - monthly data,M,Monthly,NSA,Unadjusted data (i.e. neither seasonally adjus...,TOTAL,Total,PC_ACT,Percentage of population in the labour force,T,Total,AT,Austria,2025-07,5.4
1,dataflow,ESTAT:UNE_RT_M$DEFAULTVIEW(1.0),Unemployment by sex and age - monthly data,M,Monthly,NSA,Unadjusted data (i.e. neither seasonally adjus...,TOTAL,Total,PC_ACT,Percentage of population in the labour force,T,Total,AT,Austria,2025-08,5.7
2,dataflow,ESTAT:UNE_RT_M$DEFAULTVIEW(1.0),Unemployment by sex and age - monthly data,M,Monthly,NSA,Unadjusted data (i.e. neither seasonally adjus...,TOTAL,Total,PC_ACT,Percentage of population in the labour force,T,Total,AT,Austria,2025-09,5.4
3,dataflow,ESTAT:UNE_RT_M$DEFAULTVIEW(1.0),Unemployment by sex and age - monthly data,M,Monthly,NSA,Unadjusted data (i.e. neither seasonally adjus...,TOTAL,Total,PC_ACT,Percentage of population in the labour force,T,Total,AT,Austria,2025-10,5.5
4,dataflow,ESTAT:UNE_RT_M$DEFAULTVIEW(1.0),Unemployment by sex and age - monthly data,M,Monthly,NSA,Unadjusted data (i.e. neither seasonally adjus...,TOTAL,Total,PC_ACT,Percentage of population in the labour force,T,Total,AT,Austria,2025-11,5.7


In [94]:
df_eurostat.isnull().sum()

STRUCTURE                                    0
STRUCTURE_ID                                 0
STRUCTURE_NAME                               0
freq                                         0
Time frequency                               0
s_adj                                        0
Seasonal adjustment                          0
age                                          0
Age class                                    0
unit                                         0
Unit of measure                              0
sex                                          0
Sex                                          0
geo                                          0
Geopolitical entity (reporting)              0
TIME_PERIOD                                  0
Time                                      1324
OBS_VALUE                                    0
Observation value                         1324
OBS_FLAG                                  1320
Observation status (Flag) V2 structure    1320
CONF_STATUS  

In [ ]:
## Droping unnecessary columns

df_eurostat = df_eurostat.drop(columns=['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'freq', 'Time frequency', 's_adj', 'Seasonal adjustment', 'age', 'Age class','unit', 'Unit of measure','sex','Sex' ,'Time', 'Observation value','OBS_FLAG','Observation status (Flag) V2 structure', 'CONF_STATUS','Confidentiality status (flag)',])

In [101]:
df_eurostat.dtypes

geo                                    str
Geopolitical entity (reporting)        str
TIME_PERIOD                            str
OBS_VALUE                          float64
dtype: object

In [102]:
## renaming to clean lowercase
df_eurostat.columns =['geo', 'geopolitical_entity', 'time_period', 'obs_value']

In [104]:
## Split time_period into separate year and month integer columns
df_eurostat['year'] =df_eurostat['time_period'].str[:4].astype(int)
df_eurostat['month'] = df_eurostat['time_period'].str[5:].astype(int)

In [112]:
## Drop time_period column
df_eurostat = df_eurostat.drop(columns = 'time_period')

In [113]:
# Filter to 2019 onwards
df_eurostat =df_eurostat[df_eurostat['year'] >=2019]

In [ ]:
## Export cleaned dataframe to CSV for PostgreSQL import

df_eurostat.to_csv('../data/cleaned/eurostat_unemployment_monthly_eu_2025.csv', index=False)

## 5. FRED Finland Unemployment Rate Monthly (2019–2026)
Source: Federal Reserve Economic Data (FRED) / OECD
Issues to fix:
- Column names are unclear — renaming 'observation_date' and 'LRHUTTTTFIM156S' to clean lowercase names
- Date format is '2019-01-01' — splitting into separate year and month integer columns
- Drop original date column after splitting

In [ ]:
# Load raw FRED Finland unemployment data
df_fred_unemployment = pd.read_csv('../data/raw/fred_finland_unemployment_monthly_2019_2026.csv')

In [129]:
df_fred_unemployment.head()

,observation_date,LRHUTTTTFIM156S
0,2019-01-01,6.7
1,2019-02-01,7.0
2,2019-03-01,6.5
3,2019-04-01,7.2
4,2019-05-01,6.9


In [ ]:
# Rename columns to clean lowercase names

df_fred_unemployment.columns = ['date', 'unemployment_rate']

In [ ]:
# Check data types before cleaning
df_fred_unemployment.dtypes

date                     str
unemployment_rate    float64
dtype: object

In [ ]:
# Split '2019-01-01' into separate year and month integer columns

df_fred_unemployment['year'] = df_fred_unemployment['date'].str[:4].astype(int)
df_fred_unemployment['month'] = df_fred_unemployment['date'].str[5:7].astype(int)

# Drop original date column as year and month columns replace it
df_fred_unemployment = df_fred_unemployment.drop(columns='date')

In [ ]:
# Add source column for traceability in PostgreSQL
df_fred_unemployment['source'] = 'FRED/OECD'

In [ ]:
# Export cleaned dataframe to CSV for PostgreSQL import
df_fred_unemployment.to_csv('../data/cleaned/fred_finland_unemployment_monthly_2019_2026.csv', index=False)


Exported successfully


## 6. Google Trends - yrittäjäksi (2019–2026)
Source: Google Trends
Issues to fix:
- First row is a header artifact 'Category: All categories' — skipped on load with skiprows=1
- Column names are messy — renaming to clean lowercase names
- Date format is '2019-01' — splitting into separate year and month integer columns
- Drop original date column after splitting
- Add source and search_term columns

In [137]:
df_trends_yrittajaksi=pd.read_csv('../data/raw/google_trends_yrittajaksi_2019_2026.csv', skiprows=1)

In [138]:
df_trends_yrittajaksi.head()

,Month,yrittäjäksi: (Finland)
0,2019-01,95
1,2019-02,100
2,2019-03,89
3,2019-04,66
4,2019-05,64


In [139]:
# Rename columns to clean lowercase names
df_trends_yrittajaksi.columns = ['date', 'search_interest']

In [140]:
df_trends_yrittajaksi.columns

Index(['date', 'search_interest'], dtype='str')

In [ ]:
# Split '2019-01-01' into separate year and month integer columns
df_trends_yrittajaksi['year'] = df_trends_yrittajaksi['date'].str[:4].astype(int)
df_trends_yrittajaksi['month'] =df_trends_yrittajaksi['date'].str[5:].astype(int)



,search_interest,year,month
0,95,2019,1
1,100,2019,2
2,89,2019,3
3,66,2019,4
4,64,2019,5
...,...,...,...
84,23,2026,1
85,34,2026,2
86,53,2026,3
87,30,2026,4


In [148]:
# Drop original date column as year and month columns replace it
df_trends_yrittajaksi= df_trends_yrittajaksi.drop(columns='date')

In [155]:
# Add search_term column to identify which search term each row belongs to
df_trends_yrittajaksi['search_term'] = 'yrittäjäksi'

In [144]:
# Add source column for traceability in PostgreSQL
df_trends_yrittajaksi['source'] = 'Google Trends'

In [157]:
# Export cleaned dataframe to CSV for PostgreSQL import
df_trends_yrittajaksi.to_csv('../data/cleaned/google_trends_yrittajaksi_2019_2026.csv', index=False)

## 7. Google Trends - keikkatyö (2019–2026)
Source: Google Trends
Issues to fix:
- First row is a header artifact 'Category: All categories' — skipped on load with skiprows=1
- Column names are messy — renaming to clean lowercase names
- Date format is '2019-01' — splitting into separate year and month integer columns
- Drop original date column after splitting
- Add source and search_term columns

In [152]:
df_trends_keikkayto=pd.read_csv('../data/raw/google_trends_keikkayto_2019_2026.csv',skiprows=1)

In [153]:
df_trends_keikkayto.head()

,Month,keikkatyö: (Finland)
0,2019-01,43
1,2019-02,37
2,2019-03,36
3,2019-04,31
4,2019-05,33


In [158]:
# Rename columns to clean lowercase names
df_trends_keikkayto.columns = ['date', 'search_interest']

In [159]:
# Split '2019-01-01' into separate year and month integer columns
df_trends_keikkayto['year'] = df_trends_keikkayto['date'].str[:4].astype(int)
df_trends_keikkayto['month'] = df_trends_keikkayto['date'].str[5:].astype(int)

In [160]:
# Drop original date column as year and month columns replace it
df_trends_keikkayto = df_trends_keikkayto.drop(columns='date')

In [162]:
# Add search_term column to identify which search term each row belongs to
df_trends_keikkayto['search_term'] = 'keikkatyö'

In [163]:
# Add source column for traceability in PostgreSQL
df_trends_keikkayto['source'] = 'Google Trends'

In [164]:
df_trends_keikkayto.head()

,search_interest,year,month,search_term,source
0,43,2019,1,keikkatyö,Google Trends
1,37,2019,2,keikkatyö,Google Trends
2,36,2019,3,keikkatyö,Google Trends
3,31,2019,4,keikkatyö,Google Trends
4,33,2019,5,keikkatyö,Google Trends


In [165]:
# Export cleaned dataframe to CSV for PostgreSQL import
df_trends_keikkayto.to_csv('../data/cleaned/google_trends_keikkayto_2019_2026.csv', index=False)